In [1]:
import os
import pandas as pd
import sys
sys.path.insert(0,'..')
sys.path.insert(0,'/Users/ryan/github/prosodic')
# !pip install -r /Users/ryan/github/prosodic/requirements.txt

import prosodic
pd.options.display.max_rows = 100
pd.options.display.max_columns = 100
from tqdm.auto import tqdm

import plotnine as p9
import numpy as np

constraints={
    'w_peak':1,
    'w_stress':1,
    's_unstress':1,
    'unres_across':1,
    'unres_within':1,
    # 'pentameter':1000,
    # 'iambic':1000,
}

meter = prosodic.Meter(
    constraints=constraints,
    resolve_optionality=True,
    max_s=1,
    max_w=2,
)

def get_agent(model):
    if model == 'shakespeare':
        return 'Shakespeare'
    elif model.startswith('b.'):
        cents = int(model[3:5])+1
        return f'C{cents}'
    elif 'markov' in model.lower():
        if 'shakespeare' in model.lower():
            return 'Markov (Shakespeare)'
        else:
            return 'Markov (C17-20)'
    else:
        return 'LLM'

from collections import Counter

def find_pos_in_meter_str(meter_str):
    pos=''
    posl=[]
    for ws in meter_str:
        if pos and ws!=pos[-1]:
            posl.append(pos)
            pos=''
        pos+=ws
    if pos: posl.append(pos)
    return posl

print(find_pos_in_meter_str('--+-+-+-+'))


['--', '+', '-', '+', '-', '+', '-', '+']


In [3]:
@prosodic.cache
def get_df_smpl(sample_size=500):
    df=pd.read_pickle('data.allpoems.pkl')

    def filter_poem_txt(txt):
        vparas = [para.strip() for para in txt.split('\n\n') if '\n' in para.strip()]
        lines = [l.strip() for vpara in vparas for l in vpara.split('\n') if 20>=len(l.split())>=2 and 3<=len(l.strip()) <= 100]
        return '\n'.join(lines)

    def num_lines_txt(txt):
        return len([v for v in txt.split('\n') if v.strip()])

    df['poem_txt'] = df['poem'].apply(filter_poem_txt)
    df['num_lines_txt'] = df['poem_txt'].apply(num_lines_txt)
    df['prompt'] = df['prompt'].fillna('')
    df['agent'] = df['model'].apply(get_agent)
    df = df[df.num_lines_txt>=4]
    df = df[df.num_lines_txt<=40]
    df = df.groupby('agent').head(sample_size)
    return df[['agent','model','temp','prompt','poem_txt','num_lines_txt']]

In [ ]:
df_smpl = get_df_smpl()
df_smpl

In [5]:
def parse_poem(poem_txt, force=False, parse_stash=None):
    if not force and parse_stash is not None:
        parse_df = parse_stash.get(poem_txt)
        if parse_df is not None:
            return parse_df
    
    try:
        poem = prosodic.Text(txt=poem_txt)
        parses = poem.parse(meter=meter,num_proc=4)
        parse_df = parses.best.df
        if parse_stash is not None:
            parse_stash.set(poem_txt, parse_df)
        return parse_df
    except Exception as e:
        return pd.DataFrame()
    

In [6]:
# parse_poem(df_smpl.iloc[0].poem_txt, parse_stash=prosodic.HashStash())

In [7]:
 
def get_flex_parses_df(df, force=False, fn='data.flex_parses_smpl.pkl'):
    parse_dfs = []
    if not force and os.path.exists(fn):
        parses_df = pd.read_pickle(fn)
    else:
        parse_stash = prosodic.HashStash(fn+'.db', serializer='pickle', engine='lmdb')
        parse_dfs = []
        for i,row in tqdm(df.iterrows(), desc='Parsing texts', total=len(df)):
            parse_df = parse_poem(row.poem_txt, parse_stash=parse_stash).reset_index()
            parse_df = parse_df.assign(
                agent = row.agent,
                model = row.model, 
                poem_num=i+1, 
            )
            parse_dfs.append(parse_df)

        parses_df = pd.concat(parse_dfs)
        parses_df = parses_df[['poem_num'] + [c for c in parses_df if c != 'poem_num']]
        parses_df.to_pickle(fn)

    parses_df = parses_df.query('parse_num_sylls>=4')
    return parses_df


In [ ]:
# flex_parses_df = get_flex_parses_df(force=True)
# flex_parses_df = get_flex_parses_df(df_smpl, force=False)
flex_parses_df = get_flex_parses_df(df_smpl, force=True)
# flex_parses_df

In [ ]:


def get_figdf_flex_poems(parses_df=None):
    if parses_df is None:
        parses_df = get_flex_parses_df(df_smpl, force=False)
    ld=[]
    for g,gdf in parses_df.groupby('poem_num'):
        parse_meters = ''.join(gdf.parse_meter)
        parse_meters_4thpos = gdf.parse_meter.apply(lambda x: int(x[3]=='+'))
        parse_meter_count = Counter(find_pos_in_meter_str(parse_meters))
        
        d = {'poem_num':g, 'parse_meter_4thpos':parse_meters_4thpos.mean(), 'agent':gdf.iloc[0].agent, 'model':gdf.iloc[0].model,
             'mpos_ww':parse_meter_count['--'] / parse_meter_count.total()}
        ld.append(d)
    figdf = pd.DataFrame(ld)#.merge(parses_df.drop_duplicates('poem_num'), on='poem_num', how='left')
    return figdf

figdf_flex_poems = get_figdf_flex_poems(flex_parses_df)
figdf_flex_poems

In [93]:
def plot_hood_space(figdf=None):
    if figdf is None:
        figdf = get_figdf_flex_poems().copy()

    figdf['mpos_ww']*=100
    figdf['parse_meter_4thpos']*=100

    p9.options.figure_size = (9, 8)
    title='Upper left-hand is iambic meter'
    title+='\nThe further up, the more rising the meter'
    title+='\nThe further right, the more ternary the meter'
    fig = (
        p9.ggplot(figdf, p9.aes(x="mpos_ww", y="parse_meter_4thpos", color='agent', label='agent'))
        + p9.geom_point(size=1, alpha=.5)
        + p9.geom_density_2d(size=1.25, alpha=.8)
        + p9.geom_label(size=9, data=figdf.groupby(['agent']).median(numeric_only=True).reset_index())
        # + p9.facet_wrap('agent',ncol=2)
        # + p9.facet_wrap('agent')
        + p9.theme_classic()
        + p9.theme(legend_position='none')
        + p9.scale_color_brewer(type='qual', palette='Set2')
        + p9.scale_x_continuous(limits=(0,40))
        + p9.scale_y_continuous(limits=(0,100))
        + p9.labs(y='% Lines rising', x='% Feet ternary', caption=title, title='Metrical Space')
    )
    fig.save('../../generative-humanities/Figures/HoodSpace.AllPoems.png')
    return fig


In [ ]:
plot_hood_space()